In [14]:
import os
import warnings
warnings.filterwarnings('ignore')

# Set HF token for authenticated access (optional but recommended)
# Get your token from https://huggingface.co/settings/tokens
# os.environ["HF_TOKEN"] = "your_token_here"

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load preprocessed data
print("Loading dataset...")
df = pd.read_csv("data/cleaned_arxiv_papers.csv")

print("Loading embeddings...")
doc_embeddings = np.load("data/arxiv_embeddings.npy")

print("Loading model...")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("System Ready! Embeddings shape:", doc_embeddings.shape)

Loading dataset...
Loading embeddings...
Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5280.30it/s]


System Ready! Embeddings shape: (50000, 384)


In [15]:
print(doc_embeddings.shape)
print(doc_embeddings.dtype)
doc_embeddings

(50000, 384)
float32


array([[-0.13156408, -0.00678264, -0.00367609, ...,  0.09561247,
        -0.04423941,  0.00522624],
       [-0.03589669, -0.06343268, -0.00394966, ...,  0.07206137,
        -0.13262013, -0.0041208 ],
       [-0.08129285,  0.02410299, -0.08868261, ..., -0.04524419,
        -0.0476585 , -0.01549935],
       ...,
       [-0.00546027,  0.00445373,  0.10184892, ..., -0.0455995 ,
        -0.04973711, -0.01499986],
       [ 0.03493429, -0.00508128,  0.09953675, ..., -0.01594462,
        -0.05550025, -0.03442113],
       [-0.03093104, -0.02449019, -0.00522512, ..., -0.00051519,
        -0.04664399,  0.00573538]], shape=(50000, 384), dtype=float32)



The embedding matrix is float32 with shape (50000, 384). Each row represents one paper, and the 384 columns are the semantic features learned by the model.

In [16]:
import faiss

faiss_docs = doc_embeddings.copy()
faiss.normalize_L2(faiss_docs)

print("Embeddings normalized! Ready for the FAISS Index.")

Embeddings normalized! Ready for the FAISS Index.



### **FAISS L2 Normalization & Cosine Similarity**

L2 normalization scales all vectors to unit length (magnitude of 1). This lets FAISS use its fast Inner Product function to compute cosine similarity scores between -1 and 1.

In [17]:
# FAISS index is ready for search

In [18]:
import os

index_path = "data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    search_index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(doc_embeddings)
    search_index = faiss.IndexFlatIP(384)
    search_index.add(doc_embeddings)
    
    faiss.write_index(search_index, index_path)
    print("FAISS index saved successfully!")

Loading existing FAISS index


### **FAISS Index Initialization**

IndexFlatIP creates a brute-force search index using inner product. Combined with L2 normalization, this gives us cosine similarity scores. The 384 specifies our embedding dimensionality.

### **Encoding the User Query**

Convert the user's search query into the same 384-dimensional vector space as the paper embeddings using the same model.

In [19]:
query = "deep learning for medical image analysis"
query_vec = encoder.encode([query])
print(query_vec.shape)

(1, 384)



The query is encoded using the same model and normalized to match the paper embeddings. The shape (1, 384) is ready for FAISS search.

In [20]:
query_vec = encoder.encode([query])
print(query_vec.shape)
print(query_vec)

(1, 384)
[[-3.52526270e-02  5.43267205e-02  5.65811619e-02 -4.97096591e-02
  -3.52174342e-02 -2.67348662e-02 -8.23495630e-03  1.52610037e-02
  -1.31554957e-02 -5.45781776e-02 -5.26165403e-02 -1.20118428e-02
  -8.10011551e-02  9.73842815e-02 -1.15813188e-01 -1.68417394e-02
  -9.74307582e-02 -5.52445743e-03 -1.06863678e-01  7.64726615e-03
  -4.89180721e-02 -6.45616092e-03 -3.65177169e-02  2.23113187e-02
   5.65221794e-02  4.34989482e-03  6.29342124e-02 -1.09063022e-01
   3.48554663e-02 -1.03746429e-02  7.24759325e-02 -3.63332741e-02
   1.78153962e-02  1.52815385e-02  4.68111448e-02  7.27785826e-02
  -1.72746703e-02  5.71633503e-02  6.73018489e-03  2.91875321e-02
   4.84562367e-02  5.15046231e-02  3.13239805e-02  5.97998910e-02
   8.63463804e-02 -1.21589913e-03  3.83907780e-02 -1.21435858e-02
   5.59613593e-02  9.04097706e-02 -1.46838073e-02  2.19262857e-02
  -7.75724947e-02  8.67380351e-02  3.64489332e-02 -1.94056972e-03
  -1.39828045e-02 -1.21372184e-02 -1.06874943e-01 -9.59569961e-03
 

In [21]:
faiss.normalize_L2(query_vec)
query_vec

array([[-3.52526270e-02,  5.43267205e-02,  5.65811619e-02,
        -4.97096591e-02, -3.52174342e-02, -2.67348662e-02,
        -8.23495630e-03,  1.52610037e-02, -1.31554957e-02,
        -5.45781776e-02, -5.26165403e-02, -1.20118428e-02,
        -8.10011551e-02,  9.73842815e-02, -1.15813188e-01,
        -1.68417394e-02, -9.74307582e-02, -5.52445743e-03,
        -1.06863678e-01,  7.64726615e-03, -4.89180721e-02,
        -6.45616092e-03, -3.65177169e-02,  2.23113187e-02,
         5.65221794e-02,  4.34989482e-03,  6.29342124e-02,
        -1.09063022e-01,  3.48554663e-02, -1.03746429e-02,
         7.24759325e-02, -3.63332741e-02,  1.78153962e-02,
         1.52815385e-02,  4.68111448e-02,  7.27785826e-02,
        -1.72746703e-02,  5.71633503e-02,  6.73018489e-03,
         2.91875321e-02,  4.84562367e-02,  5.15046231e-02,
         3.13239805e-02,  5.97998910e-02,  8.63463804e-02,
        -1.21589913e-03,  3.83907780e-02, -1.21435858e-02,
         5.59613593e-02,  9.04097706e-02, -1.46838073e-0

In [22]:
D, I = search_index.search(query_vec, 5)

print("Scores (D):", D)
print("Indices (I):", I)

Scores (D): [[0.72542226 0.7170659  0.71233034 0.6807244  0.6798818 ]]
Indices (I): [[26063 37161 18030 10466 29766]]


In [23]:
print(df.iloc[26063]['title'])

An overview of deep learning in medical imaging focusing on MRI


FAISS search returns similarity scores (D) and row indices (I). `k=5` retrieves the top 5 most similar papers. D contains cosine similarity scores, and I contains the corresponding DataFrame row numbers.

Now let's make a function to just return top 5 papers and we don't need to write print [df.iloc] every time

In [24]:
def search_paper(query, k=5):
    q_vec = encoder.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = search_index.search(q_vec, k)
    for score, idx in zip(scores[0], indices[0]):
        print("Similarity Score: ", score)
        print("Title: ", df.iloc[idx]['title'])
        print("Abstract:" , df.iloc[idx]['abstract'][:500])
        print()
    return scores, indices

The search function loops through results, extracting titles and abstracts from the DataFrame using the indices returned by FAISS.

In [25]:
D,I=search_paper(query)


Similarity Score:  0.72542226
Title:  An overview of deep learning in medical imaging focusing on MRI
Abstract:   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l

Similarity Score:  0.7170659
Title:  Medical Imaging with Deep Learning: MIDL 2019 -- Extended Abstract Track
Abstract:   This compendium gathers all the accepted extended abstracts from the Second
International Conference on Medical Imaging with Deep Learning (MIDL 2019),
held in London, UK, 8-10 July 2019. Note that only accepted extended abstracts
are listed here, the Proceedings 

In [26]:
import transformers
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",  # 300MB instead of 1.6GB
    device=0
)

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
type(summarizer)

In [ ]:
summary=summarizer(df.iloc[10466]['abstract'],max_length=120,min_length=40)
print(summary)
print(type(summary[0]))
print(summary[0]["summary_text"])

`.dtype` only works on NumPy arrays and Pandas Series/DataFrames. It has no meaning on a dictionary hence `AttributeError: 'dict' object has no attribute 'dtype'`. You were probably testing what type the output was, but `type(summary[0])` is the right way to check that, not `.dtype`.

### **Generating AI Summaries from Search Results**

Use a pre-trained BART model to generate concise summaries of the top search results.

In [ ]:
for score, idx in zip(D[0], I[0]):
    print("Similarity score:", score)
    print("Title:", df.iloc[idx]["title"])
    print("Abstract (Preview):", df.iloc[idx]["abstract"][:500] + "...\n")
    
    # Generate the summary using the AI pipeline
    summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40)
    
    # print(summary) # We can comment this out later, it shows the raw dictionary
    print("AI SUMMARY:")
    print(summary[0]["summary_text"])
    print("-" * 80) # Adds a nice dividing line between papers

The summarizer uses DistilBART to generate short summaries. We extract the text from the output dictionary using `summary[0]["summary_text"]`.

In [ ]:
def search_and_summarize(query, k=5):
    q_vec = encoder.encode([query])
    faiss.normalize_L2(q_vec)
    scores, indices = search_index.search(q_vec, k)
    
    for score, idx in zip(scores[0], indices[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()
        
        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
        print("AI SUMMARY:", summary[0]["summary_text"])
        print("-" * 80)

In [ ]:
search_and_summarize("Deep learning in medical science",k=5)

In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT(model=encoder)
type(kw_model)

Load KeyBERT using the same sentence transformer model for consistent embeddings. This extracts the most relevant keywords from paper abstracts.

In [ ]:
text=df.iloc[26063]['abstract']
keywords=kw_model.extract_keywords(text)
print(keywords)
print(type(keywords))
print(type(keywords[0]))

Using n-grams (1,3) to extract multi-word keywords like 'deep learning' instead of just single words.

In [ ]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None)
finalkeyword